# **SECTION 1**

In [ ]:
!pip -q install pymupdf sentence-transformers faiss-cpu gradio

# **SECTION 2**

In [ ]:
import pymupdf
import gradio as gr
import numpy as np
import faiss
import time
import os

from sentence_transformers import SentenceTransformer


# Global state for our MVP
documents = []
chunks = []
faiss_index = None
embedding_model = None

print("Imports loaded successfully.")

Imports loaded successfully.


# **SECTION 3**

In [ ]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully.


# **SECTION 4**

In [ ]:
def extract_pdf(pdf_path):
    """
    Extract text from every page of a PDF.

    Returns:
        [
            {
                "page": 1,
                "text": "..."
            },
            ...
        ]
    """

    doc = pymupdf.open(pdf_path)

    pages = []

    for page_number, page in enumerate(doc):

        text = page.get_text("text")

        pages.append({
            "page": page_number + 1,
            "text": text.strip()
        })

    doc.close()

    return pages

# **SECTION 5**

In [ ]:
def classify_document(text):
    """
    Classify a pharmaceutical document based on
    keywords found in its extracted text.
    """

    text_lower = text.lower()

    # More specific document types first

    if "certificate of quality" in text_lower:
        return "Certificate of Quality"

    elif "certificate of analysis" in text_lower:
        return "Certificate of Analysis"

    elif "bse" in text_lower or "tse" in text_lower:
        return "BSE/TSE Declaration"

    elif "packaging specification" in text_lower:
        return "Packaging Specification"

    elif (
        "to whom it may concern" in text_lower
        or "best regards" in text_lower
    ):
        return "Vendor Letter"

    elif "specification" in text_lower:
        return "Specification"

    else:
        return "Other"

# **SECTION 6**

In [ ]:
COMPLIANCE_TERMS = [
    "regulatory conformance",
    "iso 9001",
    "usp",
    "ema/410/01",
    "bio-safety",
    "bse",
    "tse"
]


def is_compliance_record(text):
    """
    Returns True if the document contains
    recognizable compliance-related terminology.
    """

    text_lower = text.lower()

    return any(
        term.lower() in text_lower
        for term in COMPLIANCE_TERMS
    )

# **SECTION 7**

In [ ]:
def chunk_text(text, chunk_size=700, overlap=100):
    """
    Split text into overlapping character-based chunks.
    """

    if not text:
        return []

    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size

        chunk = text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        if end >= len(text):
            break

        start = end - overlap

    return chunks


def build_documents(pages):
    """
    Classify pages and create smaller RAG chunks.
    """

    docs = []
    all_chunks = []

    for page in pages:

        doc_type = classify_document(page["text"])

        page_chunks = chunk_text(page["text"])

        doc = {
            "page": page["page"],
            "start_page": page["page"],
            "end_page": page["page"],
            "type": doc_type,
            "text": page["text"],
            "compliance": is_compliance_record(page["text"]),
            "chunks": len(page_chunks)
        }

        docs.append(doc)

        for chunk_number, text_chunk in enumerate(page_chunks):

            all_chunks.append({
                "page": page["page"],
                "type": doc_type,
                "chunk_id": chunk_number + 1,
                "text": text_chunk
            })

    return docs, all_chunks

# **SECTION 8**

In [ ]:
def build_vector_index(docs):
    """
    Generate embeddings for document chunks
    and store them in a FAISS vector index.
    """

    if not docs:
        return None

    texts = [
        doc["text"]
        for doc in docs
    ]

    embeddings = embedding_model.encode(
        texts,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    embeddings = embeddings.astype("float32")

    dimension = embeddings.shape[1]

    index = faiss.IndexFlatIP(dimension)

    index.add(embeddings)

    return index

# **SECTION 9**

In [ ]:
import re


def tokenize(text):
    """
    Simple tokenizer for keyword matching.
    """
    return set(
        re.findall(r"\b[a-zA-Z0-9]+\b", text.lower())
    )


def retrieve(question, docs, index, k=3):
    """
    Hybrid retrieval:
    semantic similarity + keyword overlap.
    """

    if index is None or not docs:
        return []

    # -------------------------
    # Semantic search
    # -------------------------

    query_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    # Score every chunk first
    scores, indices = index.search(
        query_embedding,
        len(docs)
    )

    # -------------------------
    # Keyword information
    # -------------------------

    question_words = tokenize(question)

    results = []

    for semantic_score, idx in zip(
        scores[0],
        indices[0]
    ):

        if idx == -1:
            continue

        doc = docs[idx]

        chunk_words = tokenize(doc["text"])

        overlap = question_words.intersection(
            chunk_words
        )

        # Percentage of query words found in chunk
        keyword_score = (
            len(overlap) / len(question_words)
            if question_words
            else 0
        )

        # Hybrid score
        final_score = (
            0.70 * float(semantic_score)
            + 0.30 * keyword_score
        )

        results.append({
            "score": final_score,
            "semantic_score": float(semantic_score),
            "keyword_score": keyword_score,
            "page": doc["page"],
            "type": doc["type"],
            "text": doc["text"]
        })

    # Highest hybrid score first
    results.sort(
        key=lambda x: x["score"],
        reverse=True
    )

    return results[:k]

# **SECTION 10**

In [ ]:
def process_document(pdf_path):
    """
    Complete document-processing pipeline.

    PDF
      -> extraction
      -> classification
      -> compliance detection
      -> chunks
      -> embeddings
      -> FAISS
      -> processing report
    """

    global documents
    global chunks
    global faiss_index

    if pdf_path is None:
        return "Please upload a PDF."

    start_time = time.time()

    # -------------------------
    # 1. Extract PDF
    # -------------------------

    pages = extract_pdf(pdf_path)

    # -------------------------
    # 2. Classify/build docs
    # -------------------------

    documents, chunks = build_documents(pages)

    # -------------------------
    # 3. Build vector database
    # -------------------------

    faiss_index = build_vector_index(chunks)

    # -------------------------
    # 4. Calculate statistics
    # -------------------------

    processing_time = time.time() - start_time

    file_name = os.path.basename(pdf_path)

    total_pages = len(pages)

    documents_found = len(documents)

    chunks_created = len(chunks)

    document_types = {
        doc["type"]
        for doc in documents
    }

    compliance_records = sum(
        1
        for doc in documents
        if doc["compliance"]
    )

    # -------------------------
    # 5. Build report
    # -------------------------

    summary = f"""
# Processing Results

**File:** {file_name}

**Pages:** {total_pages}

**Documents Found:** {documents_found}

**Chunks Created:** {chunks_created}

**Types:** {len(document_types)}

**Processing Time:** {processing_time:.2f} seconds

**Compliance Records:** {compliance_records}

---

## Document Structure

"""

    for doc in documents:

        chunk_word = (
            "chunk"
            if doc["chunks"] == 1
            else "chunks"
        )

        summary += (
            f"### {doc['type']}\n"
            f"Pages {doc['start_page']}–{doc['end_page']} "
            f"• {doc['chunks']} {chunk_word}\n\n"
        )

    return summary

# **SECTION 11**

In [ ]:
from google.colab import files

uploaded = files.upload()

##########################################

pdf_filename = next(iter(uploaded))

pdf_path = f"/content/{pdf_filename}"

result = process_document(pdf_path)

print(result)

##########################################

results = retrieve(
    "What is the lot number of the Low Flow Kit?",
    chunks,
    faiss_index,
    k=3
)

for result in results:
    print(
        f"Page: {result['page']} | "
        f"Type: {result['type']} | "
        f"Score: {result['score']:.3f}"
    )


Saving sample-sdf-document (Project 6).pdf to sample-sdf-document (Project 6) (1).pdf

# Processing Results

**File:** sample-sdf-document (Project 6) (1).pdf

**Pages:** 3

**Documents Found:** 3

**Chunks Created:** 8

**Types:** 2

**Processing Time:** 1.64 seconds

**Compliance Records:** 2

---

## Document Structure

### Vendor Letter
Pages 1–1 • 2 chunks

### Certificate of Quality
Pages 2–2 • 3 chunks

### Certificate of Quality
Pages 3–3 • 3 chunks


Page: 3 | Type: Certificate of Quality | Score: 0.581
Page: 1 | Type: Vendor Letter | Score: 0.558
Page: 1 | Type: Vendor Letter | Score: 0.478


# **SECTION 12**

In [ ]:
!pip -q install -U google-genai transformers sentencepiece

# **SECTION 13**

In [ ]:
from google.colab import userdata
from google import genai

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

client = genai.Client(api_key=GEMINI_API_KEY)

print("Gemini client ready!")

Gemini client ready!


# **SECTION 13 BACKUP**

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

fallback_model_name = "google/flan-t5-base"

fallback_tokenizer = AutoTokenizer.from_pretrained(fallback_model_name)
fallback_model = AutoModelForSeq2SeqLM.from_pretrained(fallback_model_name)

print("Local FLAN-T5 fallback ready!")

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Local FLAN-T5 fallback ready!


# **SECTION 14**

In [ ]:
import time

def answer_question(question):

    if not question.strip():
        return "Please enter a question."

    if faiss_index is None or not chunks:
        return "Please process a document first."

    start_time = time.time()

    # Retrieve top 3 relevant chunks
    results = retrieve(
        question,
        chunks,
        faiss_index,
        k=3
    )

    if not results:
        return "I couldn't find relevant information in the document."

    # Give the LLM all top-3 retrieved chunks
    context = "\n\n".join(
        [
            f"Source {i+1} - {r['type']}, Page {r['page']}:\n{r['text']}"
            for i, r in enumerate(results)
        ]
    )

    local_prompt = f"""
Answer the question using only the context below.
Give only a short factual answer.
Use the source that most directly answers the question.

Question:
{question}

Context:
{context}

Answer:
"""

    inputs = fallback_tokenizer(
        local_prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    outputs = fallback_model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=False
    )

    answer = fallback_tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    elapsed = time.time() - start_time

    sources = ", ".join(
        [f"{r['type']} Page {r['page']}" for r in results]
    )

    return f"""
### Answer

{answer}

**Retrieved Sources:** {sources}

**Top Retrieval Score:** {results[0]["score"]:.3f}

**Response Time:** {elapsed:.2f} seconds
"""

# **SECTION 15**

In [ ]:
print(
    answer_question(
        "What is the lot number of the Low Flow Kit?"
    )
)


### Answer

18356721

**Retrieved Sources:** Certificate of Quality Page 3, Vendor Letter Page 1, Vendor Letter Page 1

**Top Retrieval Score:** 0.581

**Response Time:** 8.10 seconds



# **SECTION 16**

In [ ]:
def gradio_process(file_path):
    if file_path is None:
        return "Please upload a pharmaceutical PDF."

    try:
        return process_document(file_path)

    except Exception as e:
        return f"""
# Processing Error

The document could not be processed.

**Error:** {str(e)}
"""


def gradio_chat(message, history):
    if faiss_index is None:
        return "Please upload and process a document first."

    try:
        return answer_question(message)

    except Exception as e:
        return f"Error answering question: {str(e)}"

# **SECTION 17**

In [ ]:
with gr.Blocks(title="Pharmaceutical Document Intelligence") as demo:

    gr.Markdown("""
    # 💊 Pharmaceutical Document Intelligence

    Upload a pharmaceutical PDF package to analyze its document
    structure, identify document types, and ask questions using RAG.
    """)

    with gr.Tab("📄 Document Processing"):

        pdf_upload = gr.File(
            label="Upload Pharmaceutical PDF",
            file_types=[".pdf"],
            type="filepath"
        )

        process_button = gr.Button(
            "Process Document",
            variant="primary"
        )

        processing_output = gr.Markdown()

        process_button.click(
            fn=gradio_process,
            inputs=pdf_upload,
            outputs=processing_output
        )

    with gr.Tab("💬 Document Chat"):

        gr.Markdown("""
        ### Ask questions about the processed document

        Try questions such as:
        - What is the lot number of the Low Flow Kit?
        - What is its expiration date?
        - What are the recommended storage conditions?
        - What regulatory information is provided?
        """)

        chatbot = gr.ChatInterface(
            fn=gradio_chat
        )

# **SECTION 18**

In [ ]:
# Close any previous Gradio server/tunnel
try:
    demo.close()
except:
    pass

print("Starting fresh Gradio server...")

demo.launch(
    share=True,
    debug=False,
    show_error=True,
    prevent_thread_lock=True
)

Starting fresh Gradio server...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6f6820f89bc27b16b8.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# **SECTION 19**

In [ ]:
test_questions = [
    {
        "question": "What is the lot number of the Low Flow Kit?",
        "expected_answer": "18356721",
        "expected_page": 3
    },
    {
        "question": "What is the expiration date of the Low Flow Kit?",
        "expected_answer": "20260315",
        "expected_page": 3
    },
    {
        "question": "What is the date of manufacture of the Low Flow Kit?",
        "expected_answer": "20240315",
        "expected_page": 3
    },
    {
        "question": "What is the lot number of the Gradient Flow Section?",
        "expected_answer": "17242818",
        "expected_page": 2
    },
    {
        "question": "What is the expiration date of the Gradient Flow Section?",
        "expected_answer": "20230126",
        "expected_page": 2
    },
    {
        "question": "What are the recommended storage conditions?",
        "expected_answer": "+5",
        "expected_page": 1
    },
    {
        "question": "What is the operating temperature range?",
        "expected_answer": "+2",
        "expected_page": 1
    }
]

print(f"Ground-truth test questions: {len(test_questions)}")

Ground-truth test questions: 7


# **SECTION 20**

In [ ]:
def evaluate_retrieval(test_set):

    results_table = []

    for test in test_set:

        start = time.time()

        retrieved = retrieve(
            test["question"],
            chunks,
            faiss_index,
            k=3
        )

        latency = time.time() - start

        retrieved_pages = [r["page"] for r in retrieved]

        recall_at_1 = (
            len(retrieved_pages) > 0
            and retrieved_pages[0] == test["expected_page"]
        )

        recall_at_3 = (
            test["expected_page"] in retrieved_pages
        )

        results_table.append({
            "Question": test["question"],
            "Expected Page": test["expected_page"],
            "Top Page": retrieved_pages[0] if retrieved_pages else None,
            "Recall@1": recall_at_1,
            "Recall@3": recall_at_3,
            "Latency (s)": round(latency, 3)
        })

    return results_table

# **SECTION 21**

In [ ]:
import pandas as pd

evaluation_results = evaluate_retrieval(test_questions)

df = pd.DataFrame(evaluation_results)

display(df)

recall_1 = df["Recall@1"].mean() * 100
recall_3 = df["Recall@3"].mean() * 100
avg_latency = df["Latency (s)"].mean()
error_rate = (1 - df["Recall@3"].mean()) * 100

print("\n===== RAG EVALUATION =====")
print(f"Retrieval Recall@1: {recall_1:.1f}%")
print(f"Retrieval Recall@3: {recall_3:.1f}%")
print(f"Average Retrieval Latency: {avg_latency:.3f} seconds")
print(f"Retrieval Error Rate: {error_rate:.1f}%")

,Question,Expected Page,Top Page,Recall@1,Recall@3,Latency (s)
0,What is the lot number of the Low Flow Kit?,3,3,True,True,0.075
1,What is the expiration date of the Low Flow Kit?,3,3,True,True,0.117
2,What is the date of manufacture of the Low Flo...,3,3,True,True,0.095
3,What is the lot number of the Gradient Flow Se...,2,1,False,True,0.036
4,What is the expiration date of the Gradient Fl...,2,2,True,True,0.041
5,What are the recommended storage conditions?,1,1,True,True,0.071
6,What is the operating temperature range?,1,1,True,True,0.081



===== RAG EVALUATION =====
Retrieval Recall@1: 85.7%
Retrieval Recall@3: 100.0%
Average Retrieval Latency: 0.074 seconds
Retrieval Error Rate: 0.0%


# **SECTION 22**

In [ ]:
!apt-get -qq update
!apt-get -qq install -y tesseract-ocr

print("Tesseract OCR installed.")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Tesseract OCR installed.


# **SECTION 23**

In [ ]:
def extract_pdf_with_ocr(pdf_path):
    doc = pymupdf.open(pdf_path)

    pages = []

    for page_number, page in enumerate(doc):

        # First try normal digital text extraction
        text = page.get_text("text").strip()
        extraction_method = "Digital"

        # If almost no text was found, use OCR
        if len(text) < 30:

            try:
                text_page = page.get_textpage_ocr(
                    language="eng",
                    dpi=200,
                    full=True
                )

                text = page.get_text(
                    "text",
                    textpage=text_page
                ).strip()

                extraction_method = "OCR"

            except Exception as e:
                extraction_method = "OCR Failed"

        pages.append({
            "page": page_number + 1,
            "text": text,
            "extraction_method": extraction_method
        })

    doc.close()

    return pages


print("OCR-aware extraction function ready!")

OCR-aware extraction function ready!


# **SECTION 24**

In [ ]:
test_pages = extract_pdf_with_ocr(pdf_path)

for page in test_pages:
    print(
        f"Page {page['page']} | "
        f"Method: {page['extraction_method']} | "
        f"Characters: {len(page['text'])}"
    )

Page 1 | Method: Digital | Characters: 1280
Page 2 | Method: Digital | Characters: 1362
Page 3 | Method: Digital | Characters: 1377


# **SECTION 25**

In [ ]:
def process_document(pdf_path):
    global documents
    global chunks
    global faiss_index

    if pdf_path is None:
        return "Please upload a PDF."

    start_time = time.time()

    # OCR-aware extraction
    pages = extract_pdf_with_ocr(pdf_path)

    documents, chunks = build_documents(pages)
    faiss_index = build_vector_index(chunks)

    processing_time = time.time() - start_time

    file_name = os.path.basename(pdf_path)
    total_pages = len(pages)
    documents_found = len(documents)
    chunks_created = len(chunks)
    document_types = {doc["type"] for doc in documents}
    compliance_records = sum(
        1 for doc in documents if doc["compliance"]
    )

    ocr_pages = sum(
        1 for page in pages
        if page["extraction_method"] == "OCR"
    )

    summary = f"""
# Processing Results

**File:** {file_name}

**Pages:** {total_pages}

**Documents Found:** {documents_found}

**Chunks Created:** {chunks_created}

**Types:** {len(document_types)}

**Processing Time:** {processing_time:.2f} seconds

**Compliance Records:** {compliance_records}

**OCR Pages:** {ocr_pages}

---

## Document Structure

"""

    for doc in documents:

        chunk_word = "chunk" if doc["chunks"] == 1 else "chunks"

        summary += (
            f"### {doc['type']}\n"
            f"Pages {doc['start_page']}–{doc['end_page']} "
            f"• {doc['chunks']} {chunk_word}\n\n"
        )

    return summary

# **SECTION 26**

In [ ]:
result = process_document(pdf_path)
print(result)


# Processing Results

**File:** sample-sdf-document (Project 6) (1).pdf

**Pages:** 3

**Documents Found:** 3

**Chunks Created:** 8

**Types:** 2

**Processing Time:** 2.63 seconds

**Compliance Records:** 2

**OCR Pages:** 0

---

## Document Structure

### Vendor Letter
Pages 1–1 • 2 chunks

### Certificate of Quality
Pages 2–2 • 3 chunks

### Certificate of Quality
Pages 3–3 • 3 chunks


